# 14 CNN — Exercises

Test your understanding of convolution, pooling, parameter sharing,
and receptive fields.

**Prerequisites.** Read [theory.md](theory.md) and work through
[first_principles.ipynb](first_principles.ipynb) before attempting these.

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np

%load_ext autoreload
%autoreload 2

SEED = 42
random.seed(SEED)
rng = np.random.default_rng(SEED)

## Exercise 1 — Hand Calculation: Convolution Output

Compute the convolution (cross-correlation) of a $4 \times 4$ input with a
$3 \times 3$ kernel, valid padding, stride 1.

**Input $X$:**

$$X = \begin{bmatrix}
1 & 0 & 2 & 1 \\
0 & 1 & 1 & 0 \\
2 & 0 & 0 & 1 \\
1 & 1 & 2 & 0
\end{bmatrix}$$

**Kernel $K$:**

$$K = \begin{bmatrix}
1 & 0 & -1 \\
1 & 0 & -1 \\
1 & 0 & -1
\end{bmatrix}$$

**Tasks (derive by hand, then verify numerically):**

1. What is the output shape? (Use formula: $H_{\text{out}} = H - k_h + 1$)
2. Compute each element of the output $Y$.

**Step-by-step for $Y_{0,0}$:**

$$Y_{0,0} = \sum_{m=0}^{2} \sum_{n=0}^{2} K_{m,n} \cdot X_{m,n}$$

$$= 1 \cdot 1 + 0 \cdot 0 + (-1) \cdot 2 + 1 \cdot 0 + 0 \cdot 1 + (-1) \cdot 1 + 1 \cdot 2 + 0 \cdot 0 + (-1) \cdot 0$$

$$= 1 - 2 - 1 + 2 = 0$$

**Expected results:**

| | $j=0$ | $j=1$ |
|---|---:|---:|
| $i=0$ | $0$ | $0$ |
| $i=1$ | $-1$ | $1$ |

This kernel detects vertical edges: the left column sums with $+1$ and
the right column sums with $-1$. Uniform regions give 0, edges give
nonzero values.

In [ ]:
X = np.array([[1, 0, 2, 1],
              [0, 1, 1, 0],
              [2, 0, 0, 1],
              [1, 1, 2, 0]], dtype=float)

K = np.array([[ 1,  0, -1],
              [ 1,  0, -1],
              [ 1,  0, -1]], dtype=float)

# TODO: Compute each output element by hand, then verify
# Y_00 = ...
# Y_01 = ...
# Y_10 = ...
# Y_11 = ...

# Uncomment to verify:
# Y_expected = np.array([[0, 0],
#                        [-1, 1]], dtype=float)
#
# # Cross-correlation implementation
# def cross_correlate_valid(X, K):
#     kh, kw = K.shape
#     out_h = X.shape[0] - kh + 1
#     out_w = X.shape[1] - kw + 1
#     Y = np.zeros((out_h, out_w))
#     for i in range(out_h):
#         for j in range(out_w):
#             Y[i, j] = np.sum(X[i:i+kh, j:j+kw] * K)
#     return Y
#
# Y_computed = cross_correlate_valid(X, K)
# assert np.allclose(Y_computed, Y_expected, atol=1e-10)
# print(f'Output shape: {Y_computed.shape} (4-3+1 = 2 in each dimension)')
# print(f'Output:\n{Y_computed}')
# print('Hand calculation matches. ✓')

## Exercise 2 — Coding: Implement Cross-Correlation with Padding

Implement a cross-correlation function that supports both **valid** and
**same** padding modes.

**Specifications:**

- Input: 2D array $X$ of shape $(H, W)$ and kernel $K$ of shape $(k_h, k_w)$
- `mode='valid'`: no padding, output $(H - k_h + 1) \times (W - k_w + 1)$
- `mode='same'`: zero-pad so output has same size as input ($H \times W$),
  assuming stride 1 and odd kernel size
- Stride is always 1

**Deterministic check:** Test on the given input/kernel pairs.

**Hint for same padding:** Pad by $\lfloor k_h / 2 \rfloor$ rows on top/bottom
and $\lfloor k_w / 2 \rfloor$ columns on left/right, then apply valid-mode
cross-correlation on the padded input.

In [ ]:
def cross_correlate(X, K, mode='valid'):
    """2D cross-correlation with valid or same padding.

    Args:
        X: input array, shape (H, W)
        K: kernel array, shape (kh, kw)
        mode: 'valid' or 'same'

    Returns:
        Y: output array
    """
    # TODO: implement
    # For 'same' mode: pad X with zeros, then do valid correlation
    # pad_h = K.shape[0] // 2
    # pad_w = K.shape[1] // 2
    # X_padded = np.pad(X, ((pad_h, pad_h), (pad_w, pad_w)), mode='constant')
    pass

In [ ]:
# Test cases
X_test = np.array([[1, 2, 3, 4],
                   [5, 6, 7, 8],
                   [9, 10, 11, 12],
                   [1, 2, 3, 4]], dtype=float)

K_test = np.array([[1, 0],
                   [0, -1]], dtype=float)

K_test_3x3 = np.array([[0, 1, 0],
                       [1, -4, 1],
                       [0, 1, 0]], dtype=float)  # Laplacian

# Expected: valid mode with 2x2 kernel on 4x4 input → 3x3 output
# Y[0,0] = 1*1 + 2*0 + 5*0 + 6*(-1) = 1 - 6 = -5
# Y[0,1] = 2*1 + 3*0 + 6*0 + 7*(-1) = 2 - 7 = -5
# ... all elements should be -5 (constant difference)

# TODO: uncomment and verify
# Y_valid = cross_correlate(X_test, K_test, mode='valid')
# assert Y_valid.shape == (3, 3), f'Expected (3,3), got {Y_valid.shape}'
# assert np.isclose(Y_valid[0, 0], -5.0, atol=1e-10)
# print(f'Valid mode output shape: {Y_valid.shape} ✓')
# print(f'Valid mode output:\n{Y_valid}')

# Y_same = cross_correlate(X_test, K_test_3x3, mode='same')
# assert Y_same.shape == (4, 4), f'Expected (4,4), got {Y_same.shape}'
# print(f'\nSame mode output shape: {Y_same.shape} ✓')
# print(f'Same mode output:\n{Y_same}')
# print('\nAll cross-correlation tests passed. ✓')

## Exercise 3 — Conceptual: Parameter Sharing and Receptive Field

### Part A: Why does parameter sharing make CNNs efficient?

**Questions:**

1. A dense layer connecting a $32 \times 32$ grayscale image to 64 hidden
   units needs how many parameters? A convolutional layer with 64 filters
   of size $5 \times 5$ needs how many? Compute both and find the ratio.

2. Why is parameter sharing a good inductive bias for images? What assumption
   about the data does it encode?

3. Give an example of data where parameter sharing is **harmful** (i.e., the
   CNN's assumption is violated).

### Part B: Receptive field of a 2-layer CNN

**Questions:**

4. Consider two stacked $3 \times 3$ conv layers (stride 1, no padding).
   What is the receptive field of a single unit in the output of the
   second layer? Use the formula: $r = L(k-1) + 1$.

5. How many parameters does this 2-layer stack have (single input channel,
   single output channel per layer, ignoring biases)? Compare with a single
   $5 \times 5$ filter that achieves the same receptive field.

6. Why do modern architectures (VGG, ResNet) prefer stacks of $3 \times 3$
   filters over larger filters?

In [ ]:
# Part A: Parameter count computation
# TODO: Compute and compare

# Dense layer: 32*32 inputs → 64 outputs
# dense_params = ...

# Conv layer: 64 filters of 5x5, 1 input channel
# conv_params = ...

# Uncomment to verify:
# dense_params = 32 * 32 * 64 + 64  # weights + bias
# conv_params = 64 * 1 * 5 * 5 + 64  # filters + bias
# ratio = dense_params / conv_params
# print(f'Dense parameters: {dense_params:,}')
# print(f'Conv parameters:  {conv_params:,}')
# print(f'Ratio: {ratio:.1f}×')
# print()

# Part B: Receptive field
# Two 3x3 layers, stride 1
# r = L*(k-1) + 1 = 2*(3-1) + 1 = 5

# Parameters of two 3x3 layers (1 channel each, no bias)
# two_layers = 2 * (1 * 1 * 3 * 3)  # = 18
# one_5x5 = 1 * 1 * 5 * 5           # = 25
# print(f'Receptive field of two 3×3 layers: 5×5')
# print(f'Parameters — two 3×3: {two_layers}, one 5×5: {one_5x5}')
# print(f'Two 3×3 layers use {one_5x5 - two_layers} fewer parameters')
# print(f'Plus: two 3×3 layers have two nonlinearities vs one.')

### Expected answers (reveal after attempting)

<details>
<summary>Click to reveal</summary>

**Part A:**

1. Dense: $32 \times 32 \times 64 + 64 = 65{,}600$ parameters.
   Conv: $64 \times 1 \times 5 \times 5 + 64 = 1{,}664$ parameters.
   Ratio: $\approx 39.4\times$.

2. Parameter sharing encodes the assumption that the same local pattern
   (edge, texture, corner) is useful regardless of where it appears in
   the image. This is called **translation equivariance**.

3. Shuffled-pixel images, or data where position carries unique meaning
   (e.g., a form where field 1 is always top-left and field 2 is always
   bottom-right). Here, sharing weights across positions hurts because
   different positions require different processing.

**Part B:**

4. $r = 2 \times (3-1) + 1 = 5$. Each output unit sees a $5 \times 5$
   region of the original input.

5. Two $3 \times 3$ layers: $2 \times 9 = 18$ parameters.
   One $5 \times 5$ layer: $25$ parameters. The stack is cheaper.

6. Three advantages: (a) fewer parameters for the same receptive field,
   (b) more nonlinear activations (one between each layer), increasing
   the model's representational power, and (c) easier to train with
   modern optimizers.

</details>